In [1]:
import pandas as pd
import numpy as np

Coupon Schedule

In [9]:
first_coupon_date = "2023-06-15"
maturity_date = "2033-06-15"
frequency_months = 12

In [10]:
payment_day = pd.Timestamp(first_coupon_date).day
schedule = pd.date_range(start=first_coupon_date, end=maturity_date, freq=str(frequency_months)+"MS") + pd.DateOffset(days=payment_day-1)

In [11]:
print(schedule)

DatetimeIndex(['2023-07-15', '2024-07-15', '2025-07-15', '2026-07-15',
               '2027-07-15', '2028-07-15', '2029-07-15', '2030-07-15',
               '2031-07-15', '2032-07-15'],
              dtype='datetime64[ns]', freq=None)


Cashflows

In [12]:
yearly_rate = 0.045
nominal = 1000
num_payments = len(schedule)
coupon_rate = yearly_rate*frequency_months/12
cashflows = np.zeros(num_payments)

for i in range(num_payments):
    cashflows[i] = nominal*coupon_rate

cashflows[-1] += nominal

bond = pd.DataFrame({'Cashflows':cashflows}, index=schedule)

In [13]:
print(bond)

            Cashflows
2023-07-15       45.0
2024-07-15       45.0
2025-07-15       45.0
2026-07-15       45.0
2027-07-15       45.0
2028-07-15       45.0
2029-07-15       45.0
2030-07-15       45.0
2031-07-15       45.0
2032-07-15     1045.0


Yield Curves

In [16]:
# Sample yield curve
zero_rates = np.array([0.018, 0.02, 0.021, 0.022, 0.023, 0.024, 0.025, 0.0257, 0.0265, 0.027])

Discounted Cashflows

In [17]:
discount_factors = np.zeros(num_payments)
discounted_cashflows = np.zeros(num_payments)

for i in range(num_payments):
    discount_factors[i] = 1/ (1 + zero_rates[i])
    discounted_cashflows[i] = cashflows[i] * discount_factors[i]

bond["Discount Factors"] = discount_factors
bond["Discounted Cashflows"] = discounted_cashflows

print(bond)

            Cashflows  Discount Factors  Discounted Cashflows
2023-07-15       45.0          0.982318             44.204322
2024-07-15       45.0          0.980392             44.117647
2025-07-15       45.0          0.979432             44.074437
2026-07-15       45.0          0.978474             44.031311
2027-07-15       45.0          0.977517             43.988270
2028-07-15       45.0          0.976562             43.945313
2029-07-15       45.0          0.975610             43.902439
2030-07-15       45.0          0.974944             43.872477
2031-07-15       45.0          0.974184             43.838285
2032-07-15     1045.0          0.973710           1017.526777


Price

In [19]:
price = sum(discounted_cashflows)
print(price)

1413.5012783485163


Duration

In [20]:
duration = sum(discounted_cashflows*np.arange(1,num_payments+1))/price
print(duration)

8.59745287287443


In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import numpy as np

# Define Yahoo tickers (note: some are proxy-based)
tickers = {
    "1Y": "^IRX",   # 13-week T-bill, proxy for 1Y
    "2Y": "^FVX",   # 5Y yield (Yahoo doesn't offer 2Y directly)
    "10Y": "^TNX",  # 10Y
    "30Y": "^TYX"   # 30Y
}

# Fetch recent data
data = {}
for label, ticker in tickers.items():
    df = yf.Ticker(ticker).history(period="5d")
    if not df.empty:
        data[label] = df["Close"].iloc[-1]

# Map tickers to numerical maturities
maturity_map = {"1Y": 1, "2Y": 2, "10Y": 10, "30Y": 30}
maturities = []
yields = []

for label, rate in data.items():
    if label in maturity_map:
        maturities.append(maturity_map[label])
        yields.append(rate / 100)  # percent to decimal

# Sort and interpolate
sorted_pairs = sorted(zip(maturities, yields))
maturities, yields = zip(*sorted_pairs)

curve = interp1d(maturities, yields, kind='linear', fill_value='extrapolate')

# Plot
x = np.linspace(0.5, 30, 200)
y = curve(x)

plt.plot(x, y * 100, label="Interpolated Curve")
plt.scatter(maturities, np.array(yields) * 100, color='red', label="Observed")
plt.xlabel("Maturity (Years)")
plt.ylabel("Yield (%)")
plt.title("Interpolated Yield Curve from Yahoo Finance")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()
